# Experiment 5 — ICP Hyperparameter Optimization

**Goal:** Find the best ICP configuration per cloud geometry using Optuna single-objective optimization.

The objective minimized is **mean true residual**: the mean ground-truth point-to-point
distance between the fitted transform applied to `P` and the true corresponding points in
`Q` (`T_pred(P)` vs `Q`, using the known synthetic correspondence — see
`MultiSeedSyntheticICPResult.mean_true_residuals`). This combines rotation and translation
error into one geometrically meaningful, correctly-scaled number instead of optimizing
degrees and raw distance units as separate objectives.

Rotation error, translation error, mean duration, and reliability (fraction of seeds with
rotation error < 5°) are all tracked as trial attributes for interpretation, but are not
themselves optimized.

Each trial evaluates a configuration over `TUNING_SEEDS`.
A separate study is run per cloud style, one per entry in `STYLES`.

`max_iter` and `tol` are fixed — they are stopping criteria, not quality parameters.

In [ ]:
import sys

import optuna

sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from optuna.trial import FixedTrial
from tabulate import tabulate
from optuna.importance import get_param_importances

from hpo import make_objective, build_icp_factory, build_trimmer, evaluate_icp
from visualization import OptimizationVisualizer

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid')

STYLES              = ['muscle-fiber']
N_SEEDS             = 30
N_HOLDOUT_SEEDS     = 15
N_TRIALS            = 200
MAX_ITER            = 400
TOL                 = 0.1
GEN_KWARGS          = dict(n=2000, noise_std=0.1, t_scale=8.0)
DROPOUT_PROB        = 0.2  # fraction of points randomly dropped from each observation before fitting
RELIABILITY_THRESHOLD = 0.8  # min fraction of seeds with rot err < 5° for a trial to be eligible as "best"
MULTISTART_N_JOBS   = 1  # worker processes per MultiStartICP trial; keep at 1 if study.optimize itself runs trials in parallel
STORAGE_URL         = 'sqlite:///../results/icp_hpo.db'  # persists trials so the search can be resumed across kernel restarts

# Disjoint seed sets: TUNING_SEEDS drive the search, HOLDOUT_SEEDS are only used
# afterward to validate the winning config, so reported "best config" numbers
# aren't optimistically biased by having been selected on the same data.
TUNING_SEEDS  = list(range(N_SEEDS))
HOLDOUT_SEEDS = list(range(10_000, 10_000 + N_HOLDOUT_SEEDS))

## 1. Run Studies

One single-objective Optuna study per cloud style. TPE sampler (Optuna's recommended default
for single-objective search) explores the joint space of matching strategy, feature extractor,
multi-start, trimmer, and soft-matching hyperparameters, minimizing mean true residual.

Trials are persisted to a SQLite database (`STORAGE_URL`) as they complete, keyed by `study_name`. Re-running this cell — even after a kernel restart — loads whatever trials already exist (`load_if_exists=True`) and appends `N_TRIALS` more on top, so the search can be extended incrementally instead of starting over. To start a style fresh, delete its rows from the DB or point `STORAGE_URL` elsewhere.

Study names carry a `_true_residual` suffix because earlier trials in this DB were recorded
under a 3-objective NSGA-II schema (rotation/translation error/duration) with a different
sampler — Optuna errors if you try to resume a study under a different objective shape, so
this keeps the old trials around for reference under their original study name instead of
silently discarding them.

In [ ]:
studies = {}
for style in STYLES:
    sampler = optuna.samplers.TPESampler(seed=0)
    study = optuna.create_study(
        direction='minimize',  # mean_true_residual
        sampler=sampler,
        study_name=f'icp_hpo_{style}_true_residual',
        storage=STORAGE_URL,
        load_if_exists=True,
    )
    n_before = len(study.trials)
    print(f'Running study for style={style!r} ({n_before} trial(s) already stored in {STORAGE_URL!r}) ...')
    study.optimize(
        make_objective(
            style, TUNING_SEEDS, GEN_KWARGS, MAX_ITER, TOL,
            dropout_prob=DROPOUT_PROB, multistart_n_jobs=MULTISTART_N_JOBS,
        ),
        n_trials=N_TRIALS,
        show_progress_bar=True,
    )
    studies[style] = study
    print(f'  Done. {len(study.trials)} total trials ({len(study.trials) - n_before} new). Best mean_true_residual: {study.best_value:.4g}')

Running study for style='muscle-fiber' (2 trial(s) already stored in 'sqlite:///../results/icp_hpo.db') ...


  0%|          | 0/200 [00:00<?, ?it/s]

## 2. Trial Scatter

Each point is one trial: mean true residual (the single optimized objective) vs. mean
duration, colored by matching strategy. With a single objective there's no Pareto front —
the red-outlined point is just the trial with the lowest raw mean true residual seen so far.
Section 3 picks the actual "best" config, which additionally filters on reliability.

In [ ]:
COLOR = {'hard': 'tab:blue', 'soft': 'tab:orange'}

fig, axes = plt.subplots(1, len(STYLES), figsize=(5.5 * len(STYLES), 5), squeeze=False)
for ax, style in zip(axes[0], STYLES):
    study = studies[style]
    trials = [t for t in study.trials if t.value is not None]

    x_values  = np.array([t.value for t in trials])
    y_values  = np.array([t.user_attrs['mean_duration_s'] for t in trials])
    group_ids = [t.params.get('matching', 'unknown') for t in trials]
    best_mask = np.array([t.number == study.best_trial.number for t in trials])

    OptimizationVisualizer.plot_pareto_front(
        ax, x_values, y_values, group_ids, best_mask,
        colors=COLOR,
        x_label='Mean true residual',
        y_label='Mean duration (s)',
    )
    ax.set_title(f"style='{style}'")
    ax.legend(title='matching')

fig.suptitle(
    'Trial Scatter — mean true residual vs. duration\n'
    '(red outline = lowest raw mean true residual; see Section 3 for the reliability-filtered best)'
)
plt.tight_layout()
plt.savefig('../results/5_icp_hyperparam_opt_trial_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Best Configurations

For each style, the eligible trial with the lowest mean true residual is selected as the
representative best configuration. Eligibility requires `reliability >= RELIABILITY_THRESHOLD`
(fraction of seeds with rotation error < 5°) — without this filter, a config that's usually
perfect but occasionally locks into a symmetric local minimum (a real risk for muscle-fiber's
near-periodic geometry) could still win on mean true residual alone. If no trial clears the
threshold, all trials are considered instead (with a warning).

In [ ]:
bests = {}
for style in STYLES:
    study = studies[style]
    trials = [t for t in study.trials if t.value is not None]

    reliable = [t for t in trials if t.user_attrs['reliability'] >= RELIABILITY_THRESHOLD]
    if not reliable:
        print(f"WARNING: no trial for style={style!r} reaches reliability >= {RELIABILITY_THRESHOLD:.0%}; "
              "falling back to all trials.")
        reliable = trials

    best = min(reliable, key=lambda t: t.value)
    bests[style] = best

    rows = [
        ['Mean true residual', f"{best.value:.4g}"],
        ['Rotation error (°)', f"{best.user_attrs['mean_rot_err']:.3f}"],
        ['Translation error',  f"{best.user_attrs['mean_t_err']:.3f}"],
        ['Mean duration (s)',  f"{best.user_attrs['mean_duration_s']:.3f}"],
        ['Reliability',        f"{best.user_attrs['reliability']:.0%}"],
    ] + [[k, f"{v:.4g}" if isinstance(v, float) else v] for k, v in best.params.items()]

    sigma_final = best.user_attrs.get('sigma_final')
    if sigma_final is not None:
        rows.append(['sigma_final (derived)', f'{sigma_final:.4g}'])

    print(f"\n=== Best config for style='{style}' ===")
    print(tabulate(rows, headers=['Parameter', 'Value'], tablefmt='rounded_outline'))

## 4. Parameter Importance

Importance for mean true residual, computed using **PedAnova**, which handles the conditional
parameter space (soft-matching and trimmer params are only present in trials where they were
actually suggested).

In [ ]:
ALL_PARAMS = [
    'matching', 'feature_extractor', 'fe_k', 'feature_mode', 'alpha', 'beta',
    'sigma_init', 'sigma_ratio', 'anneal_steps', 'k',
    'use_multistart', 'n_starts',
    'use_trimmer', 'trimmer_extractor', 'trimmer_fe_k', 'min_cluster_fraction',
    'eps_scaling', 'min_samples_scaling',
]

fig, axes = plt.subplots(1, len(STYLES), figsize=(6 * len(STYLES), 4), squeeze=False)

for col, style in enumerate(STYLES):
    study = studies[style]
    ax = axes[0, col]
    try:
        importance = get_param_importances(
            study,
            evaluator=optuna.importance.PedAnovaImportanceEvaluator(),
            params=ALL_PARAMS,
        )
    except Exception as e:
        ax.text(0.5, 0.5, f'N/A\n{e}', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"'{style}'")
        continue

    OptimizationVisualizer.plot_feature_importance(ax, importance, title=f"'{style}'")

fig.suptitle('Parameter importance for mean true residual, per cloud style')
plt.tight_layout()
plt.savefig('../results/5_icp_hyperparam_opt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Held-out Validation

Each `best` trial's hyperparameters were selected by minimizing mean true residual over
`TUNING_SEEDS` — the same seeds used to report the numbers in Section 3. That's an
optimistic estimate: the search explicitly picked the config that does best on exactly
that data. To get an unbiased read, replay the winning hyperparameters (via
`optuna.trial.FixedTrial`, which replays the stored `params` without a live study) on
`HOLDOUT_SEEDS`, which were never touched by the search.

In [ ]:
for style in STYLES:
    best = bests[style]
    fixed_trial = FixedTrial(best.params)

    factory = build_icp_factory(fixed_trial, MAX_ITER, TOL, multistart_n_jobs=MULTISTART_N_JOBS)
    trimmer = build_trimmer(fixed_trial, n=GEN_KWARGS.get('n', 2000))
    holdout_metrics = evaluate_icp(
        factory, style, HOLDOUT_SEEDS, GEN_KWARGS, trimmer=trimmer, dropout_prob=DROPOUT_PROB,
    )

    rows = [
        ['Mean true residual', f"{best.value:.4g}", f"{holdout_metrics['mean_true_residual']:.4g}"],
        ['Rotation error (°)', f"{best.user_attrs['mean_rot_err']:.3f}", f"{holdout_metrics['mean_rot_err']:.3f}"],
        ['Translation error',  f"{best.user_attrs['mean_t_err']:.3f}", f"{holdout_metrics['mean_t_err']:.3f}"],
        ['Mean duration (s)',  f"{best.user_attrs['mean_duration_s']:.3f}", f"{holdout_metrics['mean_duration_s']:.3f}"],
        ['Reliability',        f"{best.user_attrs['reliability']:.0%}", f"{holdout_metrics['reliability']:.0%}"],
    ]

    print(f"\n=== Held-out validation for style='{style}' ({len(HOLDOUT_SEEDS)} unseen seeds) ===")
    print(tabulate(rows, headers=['Metric', 'Tuning seeds', 'Holdout seeds'], tablefmt='rounded_outline'))